In [2]:
!pip -q install requests pyyaml python-docx
!apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package pci.ids.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package lshw.
Preparing to unpack .../2-lshw_02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../3-pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselected package usb.ids.
Preparing to unpack .../4-usb.ids_2024.03.18-1_all.deb ...
Unpacking usb.ids (2024.03.18-1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../5-

In [3]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:3b

In [4]:
import subprocess, threading, time, requests

def ollama_up():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

def watchdog():
    while True:
        if not ollama_up():
            subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "a"),
                             stderr=subprocess.STDOUT)
            time.sleep(10)
        time.sleep(3)

threading.Thread(target=watchdog, daemon=True).start()
print("watchdog running")

watchdog running


In [5]:
import subprocess, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:11434",
                  "--http-host-header", "localhost:11434"],
                 stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
!grep -o "https://[a-z0-9-]*\.trycloudflare\.com" /content/tunnel.log | head -1

https://builders-our-face-ceiling.trycloudflare.com
